# Security master tour

A guided look at the Phase 1 table, one design decision at a time. Every query here runs against the real table the build wrote, except section 5, which uses a throwaway sandbox so you can watch history being made without touching the real one.

**Rule for this notebook:** it only *calls* code from `ingestion/` and `reference_data/` — it never reimplements a merge or a reconciliation rule in a cell. If you find yourself writing pipeline logic here, it belongs in a module, with a test.

Run the build first (`python -m ingestion.build_security_master`), then run these cells top to bottom.

In [ ]:
from pyspark.sql import functions as F

from ingestion import paths, scd2
from ingestion.spark import get_spark
from reference_data.symbology import match_key, to_figi, to_yahoo

spark = get_spark("security-master-tour")
spark.sparkContext.setLogLevel("ERROR")

PATH = paths.SECURITY_MASTER
table = spark.read.format("delta").load(PATH)
current = scd2.current(spark, PATH)

## 1. The table at a glance

One row per **version** of a security, not one row per security. `is_current` marks the version that's true now. Right after the first build those are the same thing; they diverge the first time something changes.

In [ ]:
print("versions:  ", table.count())
print("securities:", current.select("security_id").distinct().count())
print("issuers:   ", current.select("issuer_id").distinct().count())

## 2. Why the key is a FIGI and not a CIK

There's one fewer issuer than securities. A **CIK** (SEC Central Index Key) identifies a *company*; a **composite FIGI** identifies a *security* — one share class, in one country. Alphabet is one company with two listed share classes:

In [ ]:
shared = current.groupBy("issuer_id").count().where("count > 1").select("issuer_id")
current.join(shared, "issuer_id").select("issuer_id", "security_id", "exchange_ticker", "security_name").show(truncate=False)

Key the table on CIK and GOOGL and GOOG collapse into one row — then a price for one gets attributed to the other. So: `security_id` (FIGI) is the row key; `issuer_id` (CIK) stays as a column, because SEC fundamentals are filed per company and Phase 5 joins on it.

## 3. One security, four spellings

Every source spells Berkshire Class B differently. `ticker_mismatch` records that the raw spellings disagree:

In [ ]:
current.where("ticker_mismatch").select(
    "exchange_ticker", "figi_ticker", "sec_ticker", "yfinance_symbol", "alpaca_symbol"
).show(truncate=False)

Two different jobs, two different treatments:

- **Matching** needs the spellings to compare equal → `match_key` folds every separator to `.`
- **Reconciliation** needs them to stay different → the flag compares raw strings

The original bug was using raw strings for *matching*: `"BRK.B" == "BRK-B"` is false, so Berkshire never found its SEC row.

In [ ]:
for spelling in ["BRK.B", "BRK-B", "BRK/B"]:
    print(f"{spelling:6} -> match_key {match_key(spelling)}")
print()
print("exchange BRK.B -> yahoo", to_yahoo("BRK.B"), "| openfigi", to_figi("BRK.B"))

## 4. The schema is a contract

Types are declared, not inferred, and Delta enforces them on every write. `NOT NULL` on the join keys means a row that could never join to anything can't get in; the `valid_interval` CHECK means history can't run backwards.

In [ ]:
table.printSchema()

In [ ]:
props = spark.sql(f"SHOW TBLPROPERTIES delta.`{PATH}`").where("key LIKE 'delta.constraints%'")
props.show(truncate=False)

## 5. Watching history get made (sandbox)

The real table has no changes yet, so this builds a sandbox copy and replays a real event: **Marsh McLennan's ticker changed from MMC to MRSH**. The FIGI didn't change — that's the point of keying on it.

It uses the exact write path the build uses (`write_security_master`), just pointed at `/tmp`.

In [ ]:
import shutil
from datetime import datetime, timezone
from ingestion.security_master_table import ATTRIBUTE_COLUMNS, write_security_master

SANDBOX = "/tmp/security_master_sandbox"
shutil.rmtree(SANDBOX, ignore_errors=True)

marsh = current.where("exchange_ticker = 'MRSH'").select(*ATTRIBUTE_COLUMNS).toPandas()
before = marsh.assign(exchange_ticker="MMC", figi_ticker="MMC", sec_ticker="MMC", yfinance_symbol="MMC")

write_security_master(spark, before, SANDBOX, datetime(2026, 1, 5, tzinfo=timezone.utc))  # first build: still MMC
write_security_master(spark, before, SANDBOX, datetime(2026, 3, 2, tzinfo=timezone.utc))  # rerun, nothing changed
write_security_master(spark, marsh, SANDBOX, datetime(2026, 6, 1, tzinfo=timezone.utc))   # rename observed

Three runs, but only two versions — the rerun changed nothing, so it wrote nothing. The old version's `valid_to` is exactly the new one's `valid_from` (half-open intervals: no gap, no overlap).

In [ ]:
spark.read.format("delta").load(SANDBOX).select(
    "security_id", "exchange_ticker", "valid_from", "valid_to", "is_current"
).orderBy("valid_from").show(truncate=False)

## 6. As-of queries — the reason this exists

"What was this security's ticker on a given date?" A backtest for April must see **MMC**, because that's what was known in April. Using today's value there is look-ahead bias — the bug Phase 5 is built to prevent.

In [ ]:
for when in ["2025-12-31", "2026-04-15", "2026-06-01", "2026-09-30"]:
    rows = scd2.as_of(spark, SANDBOX, when).select("exchange_ticker").collect()
    print(when, "->", [r.exchange_ticker for r in rows] or "(not yet observed)")

## 7. Why not just use Delta time travel?

Delta keeps every commit, and you can read any old version. But it records when **we wrote**, not when the **fact changed** — and `VACUUM` deletes old files, so it's a retention window, not a history. Look at the real table's commit log:

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{PATH}`").select("version", "timestamp", "operation", "operationMetrics.numTargetRowsInserted").show(truncate=False)

Time travel is a safety net for *mistakes* (undo a bad write). SCD2 is a model of *the world*. You want both, for different reasons.

## Try it yourself

1. In section 5, add a fourth write where `yfinance_sector` changes. How many versions are there now — and is that what you'd want for a sector reclassification?
2. Write a fifth version with an **earlier** timestamp than the last one. Which protection stops it?
3. Query `scd2.as_of` at exactly `2026-06-01`. Which version comes back, and why does the half-open interval decide it?

In [ ]:
spark.stop()